# 01 · Descarga y preparación inicial de SECOP II  Barrancabermeja



In [ ]:
# 1. Importar únicamente las librerías necesarias para este cuaderno

import os
import json
import time
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import requests
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from dotenv import load_dotenv
from IPython.display import display


In [ ]:
# 2. Detectar automáticamente la ruta raíz del proyecto

RUTA_ACTUAL = Path.cwd().resolve()

if (RUTA_ACTUAL / 'datos').exists() and (RUTA_ACTUAL / 'notebooks').exists():
    RUTA_PROYECTO = RUTA_ACTUAL
elif RUTA_ACTUAL.name.lower() == 'notebooks':
    RUTA_PROYECTO = RUTA_ACTUAL.parent
elif (RUTA_ACTUAL.parent / 'datos').exists() and (RUTA_ACTUAL.parent / 'notebooks').exists():
    RUTA_PROYECTO = RUTA_ACTUAL.parent
else:
    # Si no se reconoce la estructura, usamos la carpeta actual como raíz.
    RUTA_PROYECTO = RUTA_ACTUAL

print('🧭 Ruta raíz detectada:')
print(RUTA_PROYECTO)

In [ ]:
# 3. Definir y crear las carpetas de trabajo necesarias

RUTA_BRUTOS = RUTA_PROYECTO / 'datos' / 'brutos' / 'secop_ii'
RUTA_INTERMEDIOS = RUTA_PROYECTO / 'datos' / 'intermedios'
RUTA_PROCESADOS = RUTA_PROYECTO / 'datos' / 'procesados'
RUTA_DOCS = RUTA_PROYECTO / 'docs'

for ruta in [RUTA_BRUTOS, RUTA_INTERMEDIOS, RUTA_PROCESADOS, RUTA_DOCS]:
    ruta.mkdir(parents=True, exist_ok=True)

print('🧭 Carpetas verificadas correctamente')

## Configuración de la fuente y periodo de estudio

El corte se fija en **6 de septiembre de 2026** para que futuras actualizaciones del dataset no cambien silenciosamente el universo de este análisis.

In [ ]:
# 4. Configurar SECOP II y el periodo de análisis

DATASET_ID = 'jbjy-vk9h'
URL_API = f'https://www.datos.gov.co/resource/{DATASET_ID}.json'
URL_METADATA = f'https://www.datos.gov.co/api/views/{DATASET_ID}'

FECHA_INICIO = '2020-01-01T00:00:00.000'
FECHA_FIN_EXCLUSIVA = '2026-09-07T00:00:00.000'  # incluye hasta 2026-09-06
FECHA_CORTE = '2026-09-06'

# Token opcional de Socrata. Si no existe, el cuaderno funciona sin él.
load_dotenv(RUTA_PROYECTO / '.env')
SOCRATA_APP_TOKEN = os.getenv('SOCRATA_APP_TOKEN')

HEADERS = {}
if SOCRATA_APP_TOKEN:
    HEADERS['X-App-Token'] = SOCRATA_APP_TOKEN
    print('App Token de Socrata detectado')
else:
    print(' Sin App Token: se usará acceso público de Socrata')

print('Dataset:', DATASET_ID)
print('Periodo:', '2020-01-01 →', FECHA_CORTE)

In [ ]:
# 5. Probar la conexión y descargar los metadatos actuales del dataset

respuesta_metadata = requests.get(
    URL_METADATA,
    headers=HEADERS,
    timeout=60
)
respuesta_metadata.raise_for_status()
metadata = respuesta_metadata.json()

print('Conexión correcta con SECOP II')
print('Nombre del dataset:', metadata.get('name'))
print('Columnas reportadas:', len(metadata.get('columns', [])))

In [ ]:
# 6. Construir el diccionario de columnas reales de SECOP II

columnas_secop = pd.DataFrame([
    {
        'nombre_columna': columna.get('name'),
        'campo_api': columna.get('fieldName'),
        'tipo_dato': columna.get('dataTypeName'),
        'descripcion': columna.get('description')
    }
    for columna in metadata.get('columns', [])
])

RUTA_COLUMNAS = RUTA_DOCS / 'diccionario_columnas_secop_ii.csv'
columnas_secop.to_csv(RUTA_COLUMNAS, index=False, encoding='utf-8-sig')

print(f' Diccionario guardado en: {RUTA_COLUMNAS}')
display(columnas_secop.head(20))

In [ ]:
# 7. Seleccionar automáticamente las columnas útiles para los objetivos del proyecto

# Normalizamos el texto para que acentos y mayúsculas no afecten la selección.
def normalizar_texto(texto):
    if pd.isna(texto):
        return ''
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto)
    return texto

columnas_secop['nombre_normalizado'] = columnas_secop['nombre_columna'].map(normalizar_texto)

COLUMNAS_OBJETIVO = [
    'Nombre Entidad',
    'Nit Entidad',
    'Departamento',
    'Ciudad',
    'Localización',
    'Orden',
    'Sector',
    'Rama',
    'Entidad Centralizada',
    'Proceso de Compra',
    'ID Contrato',
    'Referencia del Contrato',
    'Estado Contrato',
    'Codigo de Categoria Principal',
    'Descripcion del Proceso',
    'Tipo de Contrato',
    'Modalidad de Contratacion',
    'Justificacion Modalidad de Contratacion',
    'Fecha de Firma',
    'Fecha de Inicio del Contrato',
    'Fecha de Fin del Contrato',
    'Condiciones de Entrega',
    'TipoDocProveedor',
    'Documento Proveedor',
    'Proveedor Adjudicado',
    'Es Grupo',
    'Es Pyme',
    'Valor del Contrato',
    'Valor de pago adelantado',
    'Valor Facturado',
    'Valor Pendiente de Pago',
    'Valor Pagado',
    'Valor Amortizado',
    'Valor Pendiente de Ejecucion',
    'URLProceso',
    'Destino Gasto',
    'Origen de los Recursos',
    'Dias adicionados'
]

objetivos_normalizados = {normalizar_texto(x) for x in COLUMNAS_OBJETIVO}

seleccion_columnas = columnas_secop[
    columnas_secop['nombre_normalizado'].isin(objetivos_normalizados)
].copy()

CAMPOS_DESCARGA = seleccion_columnas['campo_api'].dropna().tolist()

# Campos mínimos sin los cuales el análisis principal no debe continuar.
CAMPOS_MINIMOS = [
    'nombre_entidad', 'nit_entidad', 'id_contrato',
    'tipo_de_contrato', 'fecha_de_firma',
    'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato',
    'tipodocproveedor', 'documento_proveedor',
    'proveedor_adjudicado', 'valor_del_contrato'
]

faltantes_minimos = [c for c in CAMPOS_MINIMOS if c not in CAMPOS_DESCARGA]
if faltantes_minimos:
    raise ValueError(
        'Faltan campos esenciales en el dataset actual: ' + ', '.join(faltantes_minimos)
    )

print(f' Columnas seleccionadas para descarga: {len(CAMPOS_DESCARGA)}')
display(seleccion_columnas[['nombre_columna', 'campo_api', 'tipo_dato']].reset_index(drop=True))

## Catálogo de entidades locales de interés

La Alcaldía no se filtra por `ciudad`, porque SECOP II la reporta con NIT **890201900** y el campo `ciudad` puede aparecer como **No Definido**. Desde este punto, el filtro maestro es el **NIT de la entidad**.

In [ ]:
# 8. Definir el catálogo maestro de entidades del proyecto

ENTIDADES_INTERES = {
    '890201900': {
        'entidad_corta': 'Alcaldía Distrital de Barrancabermeja',
        'grupo_analisis': 'Administración central'
    },
    '829000477': {
        'entidad_corta': 'Instituto para el Fomento del Deporte y la Recreación',
        'grupo_analisis': 'Entidad local de interés'
    },
    '829001276': {
        'entidad_corta': 'Concejo de Barrancabermeja',
        'grupo_analisis': 'Entidad local de interés'
    },
    '890270833': {
        'entidad_corta': 'Empresa de Desarrollo Urbano y Vivienda',
        'grupo_analisis': 'Entidad local de interés'
    },
    '890270948': {
        'entidad_corta': 'Inspección de Tránsito y Transporte',
        'grupo_analisis': 'Entidad local de interés'
    },
    '829001855': {
        'entidad_corta': 'Personería de Barrancabermeja',
        'grupo_analisis': 'Entidad local de interés'
    },
    '8290007456': {
        'entidad_corta': 'Contraloría de Barrancabermeja',
        'grupo_analisis': 'Entidad local de interés'
    },
    '829001846': {
        'entidad_corta': 'Empresa Social del Estado Barrancabermeja',
        'grupo_analisis': 'Entidad local de interés'
    },
    '900136865': {
        'entidad_corta': 'Hospital Regional del Magdalena Medio',
        'grupo_analisis': 'Entidad local de interés'
    }
}

catalogo_entidades = pd.DataFrame([
    {
        'nit_normalizado': nit,
        'entidad_corta': info['entidad_corta'],
        'grupo_analisis': info['grupo_analisis']
    }
    for nit, info in ENTIDADES_INTERES.items()
])

print(f'Entidades definidas: {len(catalogo_entidades)}')
display(catalogo_entidades)

In [ ]:
# 9. Validar directamente cada NIT contra SECOP II

# Nit Entidad es numérico en esta vista, por eso la condición se construye sin comillas.
nits_sql = ','.join(ENTIDADES_INTERES.keys())
CONDICION_ENTIDADES = f'nit_entidad IN ({nits_sql})'

parametros_validacion = {
    '$select': 'nombre_entidad,nit_entidad,ciudad,count(*) as numero_registros',
    '$where': CONDICION_ENTIDADES,
    '$group': 'nombre_entidad,nit_entidad,ciudad',
    '$order': 'numero_registros DESC',
    '$limit': 5000
}

r = requests.get(URL_API, params=parametros_validacion, headers=HEADERS, timeout=120)
r.raise_for_status()
entidades_secop = pd.DataFrame(r.json())

if entidades_secop.empty:
    raise RuntimeError('SECOP II no devolvió registros para los NIT seleccionados.')

entidades_secop['numero_registros'] = pd.to_numeric(
    entidades_secop['numero_registros'], errors='coerce'
)
entidades_secop['nit_normalizado'] = (
    entidades_secop['nit_entidad'].astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.replace(r'\D', '', regex=True)
)

validacion_entidades = entidades_secop.merge(
    catalogo_entidades,
    on='nit_normalizado',
    how='left'
)

print(' Validación de entidades completada')
display(
    validacion_entidades[
        ['entidad_corta', 'nombre_entidad', 'nit_entidad', 'ciudad', 'numero_registros']
    ].sort_values('numero_registros', ascending=False)
)

In [ ]:
# 10. Verificar si algún NIT del catálogo no apareció en SECOP II

nits_encontrados = set(validacion_entidades['nit_normalizado'].dropna())
nits_esperados = set(ENTIDADES_INTERES.keys())
nits_faltantes = sorted(nits_esperados - nits_encontrados)

print(f' NIT esperados: {len(nits_esperados)}')
print(f' NIT encontrados: {len(nits_encontrados)}')

if nits_faltantes:
    print('⚠️ NIT del catálogo sin coincidencia:', nits_faltantes)
else:
    print(' Todos los NIT seleccionados fueron encontrados')

## Cobertura temporal antes de descargar

Esta sección cuenta registros **por fecha de firma**. Sirve para observar la cobertura real de cada entidad entre 2020 y 2026. No supone que todos los años tengan la misma exhaustividad histórica.

In [ ]:
# 11. Consultar cobertura anual por entidad usando la fecha de firma

parametros_cobertura = {
    '$select': (
        'nombre_entidad,nit_entidad,'
        'date_extract_y(fecha_de_firma) as anio,'
        'count(*) as numero_registros'
    ),
    '$where': (
        f'{CONDICION_ENTIDADES} '
        f"AND fecha_de_firma >= '{FECHA_INICIO}' "
        f"AND fecha_de_firma < '{FECHA_FIN_EXCLUSIVA}'"
    ),
    '$group': 'nombre_entidad,nit_entidad,date_extract_y(fecha_de_firma)',
    '$order': 'nit_entidad,anio',
    '$limit': 5000
}

r = requests.get(URL_API, params=parametros_cobertura, headers=HEADERS, timeout=120)
r.raise_for_status()
cobertura_anual = pd.DataFrame(r.json())

cobertura_anual['anio'] = pd.to_numeric(cobertura_anual['anio'], errors='coerce').astype('Int64')
cobertura_anual['numero_registros'] = pd.to_numeric(
    cobertura_anual['numero_registros'], errors='coerce'
)
cobertura_anual['nit_normalizado'] = (
    cobertura_anual['nit_entidad'].astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.replace(r'\D', '', regex=True)
)

mapa_nombre = {nit: info['entidad_corta'] for nit, info in ENTIDADES_INTERES.items()}
cobertura_anual['entidad_corta'] = cobertura_anual['nit_normalizado'].map(mapa_nombre)

tabla_cobertura_anual = cobertura_anual.pivot_table(
    index='entidad_corta',
    columns='anio',
    values='numero_registros',
    aggfunc='sum',
    fill_value=0
).sort_index()

print('🧭 Cobertura anual por fecha de firma')
display(tabla_cobertura_anual)

In [ ]:
# 12. Guardar la cobertura anual para documentación del proyecto

RUTA_COBERTURA = RUTA_INTERMEDIOS / '00_cobertura_anual_entidades_secop_ii.csv'
RUTA_CATALOGO = RUTA_INTERMEDIOS / '00_catalogo_entidades_interes.csv'

cobertura_anual.to_csv(RUTA_COBERTURA, index=False, encoding='utf-8-sig')
catalogo_entidades.to_csv(RUTA_CATALOGO, index=False, encoding='utf-8-sig')

print('🧭 Cobertura guardada:', RUTA_COBERTURA)
print('🧭 Catálogo guardado:', RUTA_CATALOGO)

## Descarga del universo del proyecto

Para no perder contratos cuya `Fecha de Firma` esté vacía, se incluyen también registros cuya **fecha de inicio** esté dentro del periodo, pero solo cuando la fecha de firma sea nula.

Esto preserva datos potencialmente útiles sin duplicar la lógica temporal.

In [ ]:
# 13. Construir la condición temporal final de descarga

CONDICION_PERIODO = (
    f"((fecha_de_firma >= '{FECHA_INICIO}' AND fecha_de_firma < '{FECHA_FIN_EXCLUSIVA}') "
    f"OR (fecha_de_firma IS NULL "
    f"AND fecha_de_inicio_del_contrato >= '{FECHA_INICIO}' "
    f"AND fecha_de_inicio_del_contrato < '{FECHA_FIN_EXCLUSIVA}'))"
)

CONDICION_DESCARGA = f'{CONDICION_ENTIDADES} AND {CONDICION_PERIODO}'

print('🧭 Condición final de descarga construida')

In [ ]:
# 14. Contar cuántos registros se van a descargar

parametros_total = {
    '$select': 'count(*) as total',
    '$where': CONDICION_DESCARGA
}

r = requests.get(URL_API, params=parametros_total, headers=HEADERS, timeout=120)
r.raise_for_status()
total_esperado = int(r.json()[0]['total'])

print(f'🧭 Registros esperados para descargar: {total_esperado:,}')

In [ ]:
# 15. Función robusta de descarga paginada con reintentos

def descargar_secop_paginado(
    url,
    campos,
    condicion,
    total,
    headers=None,
    limite=20000,
    max_reintentos=4
):
    bloques = []
    offset = 0
    sesion = requests.Session()

    with tqdm(total=total, desc='Descargando SECOP II', unit='reg') as barra:
        while offset < total:
            parametros = {
                '$select': ','.join(campos),
                '$where': condicion,
                '$order': 'id_contrato',
                '$limit': limite,
                '$offset': offset
            }

            datos_pagina = None

            for intento in range(1, max_reintentos + 1):
                try:
                    respuesta = sesion.get(
                        url,
                        params=parametros,
                        headers=headers or {},
                        timeout=180
                    )
                    respuesta.raise_for_status()
                    datos_pagina = respuesta.json()
                    break
                except requests.RequestException as error:
                    if intento == max_reintentos:
                        raise
                    espera = 2 ** intento
                    print(f'⚠️ Reintento {intento}/{max_reintentos} en {espera}s: {error}')
                    time.sleep(espera)

            if not datos_pagina:
                break

            bloque = pd.DataFrame(datos_pagina)
            bloques.append(bloque)
            recibidos = len(bloque)
            barra.update(recibidos)
            offset += recibidos

            if recibidos < limite:
                break

    if not bloques:
        return pd.DataFrame(columns=campos)

    return pd.concat(bloques, ignore_index=True)

In [ ]:
# 16. Descargar los contratos de las entidades seleccionadas

secop_bruto = descargar_secop_paginado(
    url=URL_API,
    campos=CAMPOS_DESCARGA,
    condicion=CONDICION_DESCARGA,
    total=total_esperado,
    headers=HEADERS,
    limite=20000
)

print(f'🧭 Registros descargados: {len(secop_bruto):,}')

if len(secop_bruto) != total_esperado:
    print(
        f'⚠️ El conteo inicial era {total_esperado:,} y se descargaron '
        f'{len(secop_bruto):,}. El dataset puede haber cambiado durante la descarga.'
    )
else:
    print('🧭 La descarga coincide con el conteo esperado')

display(secop_bruto.head())

In [ ]:
# 17. Guardar inmediatamente una copia BRUTA sin transformar

RUTA_BRUTO_PARQUET = RUTA_BRUTOS / 'secop_ii_entidades_interes_2020_2026-09-06.parquet'
RUTA_BRUTO_CSV = RUTA_BRUTOS / 'secop_ii_entidades_interes_2020_2026-09-06.csv'

secop_bruto.to_parquet(RUTA_BRUTO_PARQUET, index=False)
secop_bruto.to_csv(RUTA_BRUTO_CSV, index=False, encoding='utf-8-sig')

print('🧭 Copia bruta Parquet:', RUTA_BRUTO_PARQUET)
print('🧭 Copia bruta CSV:', RUTA_BRUTO_CSV)

## Base inicial para los siguientes cuadernos

A partir de aquí **no eliminamos registros**. Solo añadimos variables auxiliares para identificar entidad, fechas y periodo político de referencia. La limpieza de CPS, personas naturales, duplicados y duración se hará en el siguiente cuaderno.

In [ ]:
# 18. Crear una base inicial con variables auxiliares sin alterar la copia bruta

base_inicial = secop_bruto.copy()

base_inicial['nit_normalizado'] = (
    base_inicial['nit_entidad'].astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.replace(r'\D', '', regex=True)
)

mapa_grupo = {nit: info['grupo_analisis'] for nit, info in ENTIDADES_INTERES.items()}
base_inicial['entidad_corta'] = base_inicial['nit_normalizado'].map(mapa_nombre)
base_inicial['grupo_analisis'] = base_inicial['nit_normalizado'].map(mapa_grupo)
base_inicial['es_alcaldia'] = base_inicial['nit_normalizado'].eq('890201900')

for campo_fecha in [
    'fecha_de_firma',
    'fecha_de_inicio_del_contrato',
    'fecha_de_fin_del_contrato'
]:
    if campo_fecha in base_inicial.columns:
        base_inicial[campo_fecha] = pd.to_datetime(
            base_inicial[campo_fecha], errors='coerce'
        )

base_inicial['fecha_referencia'] = base_inicial['fecha_de_firma'].fillna(
    base_inicial['fecha_de_inicio_del_contrato']
)

base_inicial['fecha_referencia_origen'] = np.where(
    base_inicial['fecha_de_firma'].notna(),
    'fecha_firma',
    'fecha_inicio_sin_firma'
)

base_inicial['anio_referencia'] = base_inicial['fecha_referencia'].dt.year.astype('Int64')
base_inicial['mes_referencia'] = base_inicial['fecha_referencia'].dt.month.astype('Int64')

base_inicial['periodo_referencia'] = np.select(
    [
        base_inicial['fecha_referencia'].between('2020-01-01', '2023-12-31 23:59:59'),
        base_inicial['fecha_referencia'].between('2024-01-01', '2026-09-06 23:59:59')
    ],
    [
        'Alfonso Eljach (2020-2023)',
        'Jonathan Vásquez (2024-2026 corte)'
    ],
    default='Fuera del periodo'
)

# La etiqueta de periodo es una referencia temporal para entidades complementarias;
# NO significa que sus contratos sean contratos del alcalde.

if 'valor_del_contrato' in base_inicial.columns:
    base_inicial['valor_contrato_num'] = pd.to_numeric(
        base_inicial['valor_del_contrato'], errors='coerce'
    )

print('🧭 Variables auxiliares creadas')
display(
    base_inicial[
        [
            'entidad_corta', 'es_alcaldia', 'id_contrato',
            'fecha_de_firma', 'fecha_de_inicio_del_contrato',
            'fecha_referencia_origen', 'periodo_referencia'
        ]
    ].head(10)
)

In [ ]:
# 19. Diagnóstico básico de integridad de la descarga

registros = len(base_inicial)
contratos_unicos = base_inicial['id_contrato'].nunique(dropna=True)
duplicados_id = base_inicial['id_contrato'].duplicated(keep=False).sum()
sin_id = base_inicial['id_contrato'].isna().sum()
sin_fecha_firma = base_inicial['fecha_de_firma'].isna().sum()

print(f'🧭 Filas descargadas: {registros:,}')
print(f'🧭 ID de contrato únicos: {contratos_unicos:,}')
print(f'⚠️ Filas involucradas en ID duplicado: {duplicados_id:,}')
print(f'⚠️ Registros sin ID contrato: {sin_id:,}')
print(f'⚠️ Registros sin fecha de firma: {sin_fecha_firma:,}')

In [ ]:
# 20. Crear un resumen por entidad y periodo político de referencia

resumen_inicial = (
    base_inicial
    .groupby(['entidad_corta', 'periodo_referencia'], dropna=False)
    .agg(
        registros=('id_contrato', 'size'),
        contratos_unicos=('id_contrato', 'nunique'),
        proveedores_unicos=('documento_proveedor', 'nunique'),
        primera_fecha=('fecha_referencia', 'min'),
        ultima_fecha=('fecha_referencia', 'max')
    )
    .reset_index()
)

print('🧭 Resumen inicial del universo descargado')
display(resumen_inicial)

In [ ]:
# 21. Crear cobertura mensual local a partir de la descarga

cobertura_mensual = (
    base_inicial
    .dropna(subset=['fecha_referencia'])
    .assign(mes_calendario=lambda x: x['fecha_referencia'].dt.to_period('M').astype(str))
    .groupby(['entidad_corta', 'mes_calendario'])
    .agg(
        registros=('id_contrato', 'size'),
        contratos_unicos=('id_contrato', 'nunique')
    )
    .reset_index()
)

display(cobertura_mensual.head(20))

In [ ]:
# 22. Guardar la base inicial y los resúmenes para los siguientes notebooks

RUTA_BASE_INICIAL = RUTA_INTERMEDIOS / '01_secop_ii_base_inicial.parquet'
RUTA_RESUMEN = RUTA_INTERMEDIOS / '01_resumen_inicial_entidades.csv'
RUTA_COBERTURA_MENSUAL = RUTA_INTERMEDIOS / '01_cobertura_mensual_entidades.csv'

base_inicial.to_parquet(RUTA_BASE_INICIAL, index=False)
resumen_inicial.to_csv(RUTA_RESUMEN, index=False, encoding='utf-8-sig')
cobertura_mensual.to_csv(RUTA_COBERTURA_MENSUAL, index=False, encoding='utf-8-sig')

print('🧭 Base inicial:', RUTA_BASE_INICIAL)
print('🧭 Resumen:', RUTA_RESUMEN)
print('🧭 Cobertura mensual:', RUTA_COBERTURA_MENSUAL)

In [ ]:
# 23. Guardar un manifiesto de reproducibilidad de la descarga

manifiesto = {
    'dataset_id': DATASET_ID,
    'url_api': URL_API,
    'fecha_inicio': '2020-01-01',
    'fecha_corte_inclusive': FECHA_CORTE,
    'fecha_ejecucion': datetime.now().isoformat(timespec='seconds'),
    'registros_descargados': int(len(secop_bruto)),
    'campos_descargados': CAMPOS_DESCARGA,
    'nits_entidades': list(ENTIDADES_INTERES.keys()),
    'entidades': {nit: info['entidad_corta'] for nit, info in ENTIDADES_INTERES.items()},
    'nota_metodologica': (
        'La Alcaldía se identifica por NIT 890201900, no por ciudad. '
        'Las demás entidades se conservan como universos complementarios y no se atribuyen automáticamente al alcalde.'
    )
}

RUTA_MANIFIESTO = RUTA_BRUTOS / 'manifest_descarga_secop_ii.json'
with open(RUTA_MANIFIESTO, 'w', encoding='utf-8') as f:
    json.dump(manifiesto, f, ensure_ascii=False, indent=2)

print('🧭 Manifiesto guardado:', RUTA_MANIFIESTO)

## ✅ Fin del cuaderno 01

Si todas las celdas terminaron sin error, ya quedan creados:

- `datos/brutos/secop_ii/secop_ii_entidades_interes_2020_2026-09-06.parquet`
- `datos/brutos/secop_ii/secop_ii_entidades_interes_2020_2026-09-06.csv`
- `datos/brutos/secop_ii/manifest_descarga_secop_ii.json`
- `datos/intermedios/00_catalogo_entidades_interes.csv`
- `datos/intermedios/00_cobertura_anual_entidades_secop_ii.csv`
- `datos/intermedios/01_secop_ii_base_inicial.parquet`
- `datos/intermedios/01_resumen_inicial_entidades.csv`
- `datos/intermedios/01_cobertura_mensual_entidades.csv`

El siguiente cuaderno será **`02_limpieza_y_construccion_cps.ipynb`**, donde identificaremos CPS, personas naturales, duración, valor mensual, duplicados y contratistas recurrentes.